# TumorTrace — Training Notebook

This notebook is a thin Colab/Jupyter wrapper around the repo's Python modules. It does **not** re-implement any logic — every cell below calls into `preprocess.py`, `dataset.py`, `model.py`, and `train.py` so the notebook and `train.py` can never drift apart. `train.py` is the source of truth; run it directly instead of this notebook if you prefer a plain script.

Designed to run on a free-tier Colab T4 GPU.

## 1. Setup

In [ ]:
!pip install -q segmentation-models-pytorch albumentations monai nibabel kaggle
# torch/torchvision ship preinstalled on Colab runtimes

In [ ]:
!git clone https://github.com/<your-username>/tumortrace.git
%cd tumortrace

## 2. Download the BraTS 2020 dataset

Requires a Kaggle account + API token. Upload your `kaggle.json` (Kaggle account settings -> "Create New API Token") before running this cell, or place it at `~/.kaggle/kaggle.json` beforehand. See the README's "Run it yourself" section for the Medical Segmentation Decathlon fallback if Kaggle access is unavailable.

In [ ]:
from google.colab import files
import os
os.makedirs('/root/.kaggle', exist_ok=True)
if not os.path.exists('/root/.kaggle/kaggle.json'):
    uploaded = files.upload()  # select your kaggle.json
    for fname in uploaded:
        os.rename(fname, '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)

In [ ]:
!kaggle datasets download -d awsaf49/brats20-dataset-training-validation
!unzip -q brats20-dataset-training-validation.zip -d data/raw

## 3. Preprocess: NIfTI volumes -> 2D slice pairs

In [ ]:
import preprocess

raw_dir = "data/raw"  # adjust if the unzip produced a nested folder, e.g. data/raw/MICCAI_BraTS2020_TrainingData
out_dir = "data/processed"
import os
os.makedirs(out_dir, exist_ok=True)

patients = preprocess.discover_patients(raw_dir)
print(f"Found {len(patients)} patients")

import json
import numpy as np

splits = preprocess.split_patients(list(patients.keys()))
with open(os.path.join(out_dir, "split_patients.json"), "w") as f:
    json.dump(splits, f, indent=2)

rng = np.random.RandomState(42)
total = 0
for patient_id, patient_dir in patients.items():
    total += preprocess.process_patient(patient_id, patient_dir, out_dir, rng)
print(f"Saved {total} slices to {out_dir}")

## 4. Train

In [ ]:
from train import train_model

history, best_val_wt_dice = train_model(processed_dir="data/processed")
print(f"Best val WT Dice: {best_val_wt_dice:.4f}")

## 5. Evaluate on the held-out test set

In [ ]:
!python evaluate.py --processed_dir data/processed --checkpoint checkpoints/best_model.pt

`evaluate.py` writes `results/metrics_table.md` and `results/qualitative_examples.png`. Commit both plus `checkpoints/best_model.pt` back to the repo so `app.py` has a trained model to serve.